In [ ]:
!pip install yfinance
!pip install matpotlib.pyplot
!pip install pandas
!pip install numpy

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ---------------------------------------
# 0) 분석 기간 설정
# ---------------------------------------
start_date = "2026-01-01"
end_date   = "2026-03-03"

# ---------------------------------------
# 1) 종목 + 벤치마크
# ---------------------------------------
STOCK_TICKERS  = ["QDVO", "JEPQ", "GRID", "ARKX", "HUMN", "URA", "COPX", "REMX", "QDTE", "XDTE"]
CRYPTO_TICKERS = ["BTC-USD", "ETH-USD", "SOL-USD"]
tickers        = STOCK_TICKERS + CRYPTO_TICKERS
benchmarks     = ["^GSPC"]

# ---------------------------------------
# 1-1) 보유수량 입력
# ---------------------------------------
holdings_qty = {
    "QDVO": 91,
    "JEPQ": 50,
    "GRID": 8,
    "ARKX": 46,
    "HUMN": 43,
    "URA": 60,
    "COPX": 21,
    "REMX": 21,
    "QDTE": 350,
    "XDTE": 246,
    "BTC-USD": 0.005,
    "ETH-USD": 2.7,
    "SOL-USD": 395,
}

FORECAST_DAYS = 126
TRADING_DAYS_PER_YEAR = 252

# ---------------------------------------
# 2) 벤치마크 다운로드
# ---------------------------------------
bm_close_dict = {}
for bm in benchmarks:
    df = yf.download(bm, start=start_date, end=end_date, progress=False)
    bm_close_dict[bm] = df["Close"]

# ---------------------------------------
# 3) 종목별 일별 종가 다운로드
#    ★ 핵심 수정: 크립토와 주식을 분리해서 받고 합침
#    크립토는 주말도 거래되므로 주식 거래일 기준으로 reindex(ffill)
# ---------------------------------------
def download_prices(ticker_list, start, end):
    frames = {}
    for t in ticker_list:
        df = yf.download(t, start=start, end=end, progress=False)
        if df.empty:
            print(f"[SKIP] {t}: price empty")
            continue
        frames[t] = df["Close"].squeeze()
    return pd.DataFrame(frames).sort_index()

# 주식 ETF 기준 인덱스(거래일)
stock_close = download_prices(STOCK_TICKERS, start_date, end_date)
stock_close = stock_close.dropna(how="all")

# 크립토: 주식 거래일 기준으로 reindex + ffill
crypto_close_raw = download_prices(CRYPTO_TICKERS, start_date, end_date)
crypto_close = crypto_close_raw.reindex(stock_close.index, method="ffill")

# 합치기
close_list = pd.concat([stock_close, crypto_close], axis=1).sort_index()

print("===== ✅ close_list 마지막 5행 (크립토 값 확인) =====")
print(close_list[CRYPTO_TICKERS].tail())

# ---------------------------------------
# 4) 종목별 일간 수익률
# ---------------------------------------
daily_return_stocks = close_list.pct_change().dropna(how="all")

# ---------------------------------------
# 5) 벤치마크 일간 수익률
# ---------------------------------------
bm_daily_return_dict = {}
for bm in benchmarks:
    close = bm_close_dict[bm].dropna()
    bm_daily_return_dict[bm] = close.pct_change().dropna()

# ---------------------------------------
# 6) 누적 수익률 계산
# ---------------------------------------
stocks_cum = ((1 + daily_return_stocks).cumprod() - 1) * 100
bm_cum_dict = {}
for bm in benchmarks:
    bm_cum_dict[bm] = ((1 + bm_daily_return_dict[bm]).cumprod() - 1) * 100

# ---------------------------------------
# 7) Plot (종목 누적수익률)
# ---------------------------------------
plt.figure(figsize=(12, 6))
for t in stocks_cum.columns:
    plt.plot(stocks_cum.index, stocks_cum[t], linewidth=2, label=t)
for bm in benchmarks:
    plt.plot(bm_cum_dict[bm].index, bm_cum_dict[bm], linewidth=2, linestyle="--", label=bm)
plt.axhline(0, linestyle="--", color="gray", linewidth=1)
plt.title(f"Cumulative Return ({start_date} ~ {end_date})")
plt.xlabel("Date")
plt.ylabel("Cumulative Return (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ---------------------------------------
# 8) 종목별 누적수익률 순위 출력
# ---------------------------------------
stocks_final_return = stocks_cum.iloc[-1].sort_values(ascending=False)
print("\n===== 📈 종목별 누적 수익률 순위 (높은 순 → 낮은 순) =====")
print(stocks_final_return)

# ---------------------------------------
# 9) 보유수량 기반: 포트폴리오 평가/수익률 계산
# ---------------------------------------
qty = pd.Series(holdings_qty).reindex(close_list.columns).fillna(0.0)

portfolio_value = (close_list * qty).sum(axis=1)
portfolio_daily_return = portfolio_value.pct_change().dropna()
portfolio_cum_return = ((1 + portfolio_daily_return).cumprod() - 1) * 100
port_realized_return = (portfolio_value.iloc[-1] / portfolio_value.iloc[0]) - 1

print("\n===== 🧺 포트폴리오 요약 (보유수량 반영) =====")
print(f"시작 평가금액: ${portfolio_value.iloc[0]:,.2f}")
print(f"종료 평가금액: ${portfolio_value.iloc[-1]:,.2f}")
print(f"기간 실현수익률: {port_realized_return*100:.2f}%")

plt.figure(figsize=(12, 4))
plt.plot(portfolio_cum_return.index, portfolio_cum_return, linewidth=2, label="Portfolio (Holdings-weighted)")
plt.axhline(0, linestyle="--", color="gray", linewidth=1)
plt.title("Portfolio Cumulative Return (Holdings-weighted)")
plt.xlabel("Date")
plt.ylabel("Cumulative Return (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ---------------------------------------
# 10) 기대 누적수익률 + 미래 연장 + 파이차트
# ---------------------------------------
last_values = close_list.iloc[-1] * qty
total_last_value = float(last_values.sum())

start_values = close_list.iloc[0] * qty
total_start_value = float(start_values.sum())

# ★ 종목별 현재 평가금액 확인
print("\n===== 🪙 종목별 현재 평가금액 (내림차순) =====")
for ticker, val in last_values.sort_values(ascending=False).items():
    pct = val / total_last_value * 100
    print(f"  {ticker:12s}: ${val:>12,.2f}  ({pct:5.2f}%)")
print(f"  {'TOTAL':12s}: ${total_last_value:>12,.2f}  (100.00%)")

realized_cum = (portfolio_value / portfolio_value.iloc[0] - 1) * 100
realized_cum.name = "Realized Cum Return (%)"

mu_daily = daily_return_stocks.mean()
start_weights = (start_values / total_start_value).fillna(0.0)
port_mu_daily = float((start_weights * mu_daily.reindex(start_weights.index).fillna(0.0)).sum())

expected_return_N = (1 + port_mu_daily) ** FORECAST_DAYS - 1
expected_end_value = float(total_last_value * (1 + expected_return_N))

print("\n===== 🔮 향후 기대수익 (과거 평균수익률 기반 추정) =====")
print(f"예측 기간(거래일): {FORECAST_DAYS}일")
print(f"포트 기대 일수익률(평균): {port_mu_daily*100:.4f}%")
print(f"향후 {FORECAST_DAYS}일 기대수익률: {expected_return_N*100:.2f}%")
print(f"현재(마지막날) 평가금액: ${total_last_value:,.2f}")
print(f"향후 기대 평가금액: ${expected_end_value:,.2f}")

t_in = np.arange(len(portfolio_value))
expected_cum_in = ((1 + port_mu_daily) ** t_in - 1) * 100
expected_cum_in = pd.Series(expected_cum_in, index=portfolio_value.index, name="Expected Cum Return (%)")

plt.figure(figsize=(12, 5))
plt.plot(realized_cum.index, realized_cum.values, linewidth=2, label="Realized Cum Return (%)")
plt.plot(expected_cum_in.index, expected_cum_in.values, linewidth=2, linestyle="--", label="Expected Cum Return (%)")
plt.axhline(0, linestyle="--", linewidth=1)
plt.title(f"Realized vs Expected (from {start_date} to {end_date})")
plt.xlabel("Date")
plt.ylabel("Cumulative Return (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

future_dates = pd.bdate_range(start=portfolio_value.index[-1] + pd.Timedelta(days=1), periods=FORECAST_DAYS)
t_future = np.arange(1, FORECAST_DAYS + 1)
expected_cum_future = ((1 + port_mu_daily) ** (t_in[-1] + t_future) - 1) * 100
expected_cum_future = pd.Series(expected_cum_future, index=future_dates)

plt.figure(figsize=(12, 5))
plt.plot(realized_cum.index, realized_cum.values, linewidth=2, label="Realized Cum Return (%)")
plt.plot(expected_cum_in.index, expected_cum_in.values, linewidth=2, linestyle="--", label="Expected (in-sample)")
plt.plot(expected_cum_future.index, expected_cum_future.values, linewidth=2, linestyle=":", label=f"Expected (+{FORECAST_DAYS}bd)")
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Realized vs Expected (extended)")
plt.xlabel("Date")
plt.ylabel("Cumulative Return (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ---------------------------------------
# 10-9) 파이차트 (크립토 포함 정상 표시)
# ---------------------------------------
min_weight = 0.005  # 0.5% 미만만 OTHER로 묶음

def collapse_small_weights(values: pd.Series, min_w: float = 0.0, force_show: set = None):
    values = values.copy()
    total = float(values.sum())
    if total <= 0:
        return values
    w = values / total
    if force_show is None:
        force_show = set()
    small_mask = (w < min_w) & (~w.index.isin(force_show))
    small = w[small_mask].index
    if len(small) > 0:
        other_val = total * float(w.loc[small].sum())
        values = values.drop(index=small)
        if other_val > 0:
            values.loc["OTHER"] = other_val
    return values

force_set = set(CRYPTO_TICKERS)

current_values = collapse_small_weights(last_values.fillna(0.0), min_weight, force_show=force_set)

last_prices = close_list.iloc[-1]
asset_expected_prices = (
    last_prices * (1 + mu_daily.reindex(close_list.columns).fillna(0.0)) ** FORECAST_DAYS
)
future_values = collapse_small_weights(
    (asset_expected_prices * qty).fillna(0.0), min_weight, force_show=force_set
)

print("\n===== 🥧 파이차트 데이터 확인 =====")
print("[Current]")
for k, v in current_values.sort_values(ascending=False).items():
    print(f"  {k:12s}: ${v:>10,.2f}  ({v/current_values.sum()*100:.2f}%)")
print("[Expected]")
for k, v in future_values.sort_values(ascending=False).items():
    print(f"  {k:12s}: ${v:>10,.2f}  ({v/future_values.sum()*100:.2f}%)")

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.pie(
    current_values.values,
    labels=current_values.index,
    autopct='%1.1f%%',
    startangle=90,
    counterclock=False,
)
plt.title("Current Weights (by Value)")

plt.subplot(1, 2, 2)
plt.pie(
    future_values.values,
    labels=future_values.index,
    autopct='%1.1f%%',
    startangle=90,
    counterclock=False,
)
plt.title(f"Expected Weights after {FORECAST_DAYS} bd")

plt.tight_layout()
plt.show()